<a href="https://colab.research.google.com/github/Nithu006/J26-IT-388/blob/main/Reserch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("/addiction_2019_features_tfidf_256.csv")

print(df.shape)                 # rows, columns
print(list(df.columns[:15]))    # mudhal 15 column names

keys = ["author", "user", "date", "time", "utc", "subreddit", "comment", "score"]
print([c for c in df.columns if any(k in c.lower() for k in keys)])

df.iloc[:3, :8]

(1257, 350)
['subreddit', 'author', 'date', 'post', 'automated_readability_index', 'coleman_liau_index', 'flesch_kincaid_grade_level', 'flesch_reading_ease', 'gulpease_index', 'gunning_fog_index', 'lix', 'smog_index', 'wiener_sachtextformel', 'n_chars', 'n_long_words']
['subreddit', 'author', 'date', 'liwc_time', 'tfidf_time']


,subreddit,author,date,post,automated_readability_index,coleman_liau_index,flesch_kincaid_grade_level,flesch_reading_ease
0,addiction,Lillie1990,2019/01/01,Quitting coke and need advice So I’m 20. I’ve ...,2.315272,4.091255,3.580870,89.872283
1,addiction,P3nw00dx,2019/01/01,I am addicted to sleep I really really like to...,1.591231,2.547960,3.821538,90.818462
2,addiction,throwaway146900,2019/01/01,Addicted to scratching my balls This may sound...,2.829911,4.762222,3.912143,87.370000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
import pandas as pd

users = pd.read_csv("/content/users_annonym.csv")
tl = pd.read_csv("/content/timelines_annonym.csv")

print("USERS:", users.shape)
print(users.columns.tolist())
print(users.head(3))

print("TIMELINES:", tl.shape)
print(tl.columns.tolist())
print(tl.head(3))

FileNotFoundError: [Errno 2] No such file or directory: '/content/users_annonym.csv'

In [2]:
import pandas as pd

users = pd.read_csv("/users_annonym.csv")
tl = pd.read_csv("/timelines_annonym.csv")

print("USERS:", users.shape)
print(users.columns.tolist())
print(users.head(3))

print("TIMELINES:", tl.shape)
print(tl.columns.tolist())
print(tl.head(3))

USERS: (165, 6)
['Unnamed: 0', 'participant_id', 'gold_depression', 'gold_anxiety', 'depression', 'anxiety']
   Unnamed: 0 participant_id gold_depression gold_anxiety  depression  anxiety
0           0  participant_0       depressed      anxious           1        1
1           1  participant_1       depressed         none           1        0
2           2  participant_2            none         none           0        0
TIMELINES: (6420, 30)
['Unnamed: 0', 'participant_id', 'created_date', 'anonymised_text', 'admiration', 'amusement', 'anger', 'annoyance', 'anticipation', 'approval', 'confusion', 'curiosity', 'desire', 'disapproval', 'disgust', 'disappointed', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realisation', 'remorse', 'sadness', 'surprise', 'trust']
   Unnamed: 0 participant_id               created_date  \
0           0  participant_0  2023-03-09 05:17:54+00:00   
1           1  participant_0  2023-04-20 04:24:13+01:00   


In [4]:
USER_COL = "participant_id"
TIME_COL = "created_date"
LABEL_COL = "depression"

users = users.drop(columns=["Unnamed: 0"], errors="ignore")
tl = tl.drop(columns=["Unnamed: 0"], errors="ignore")

print("Before:", tl.shape)

tl = tl.drop_duplicates()
tl = tl.dropna(subset=[USER_COL, TIME_COL])
tl[TIME_COL] = pd.to_datetime(tl[TIME_COL], errors="coerce", utc=True)   # mixed +00:00/+01:00 handle aagum
tl = tl.dropna(subset=[TIME_COL])
tl = tl.sort_values([USER_COL, TIME_COL]).reset_index(drop=True)

print("After:", tl.shape)
print("Date range:", tl[TIME_COL].min(), "to", tl[TIME_COL].max())
print("Users:", tl[USER_COL].nunique())

Before: (6420, 29)
After: (6420, 29)
Date range: 2017-09-27 04:39:55+00:00 to 2023-12-09 12:20:25+00:00
Users: 165


In [5]:
counts = tl[USER_COL].value_counts()
print(counts.describe())

span = tl.groupby(USER_COL)[TIME_COL].agg(lambda s: (s.max() - s.min()).days)
print(span.describe())     # oru user-oda tweets ethanai naal span


count    165.000000
mean      38.909091
std      116.233565
min        1.000000
25%        1.000000
50%        5.000000
75%       21.000000
max      940.000000
Name: count, dtype: float64
count     165.000000
mean       87.321212
std       255.716497
min         0.000000
25%         0.000000
50%        17.000000
75%        70.000000
max      2256.000000
Name: created_date, dtype: float64


In [6]:
df = tl.merge(users[[USER_COL, "depression", "anxiety"]], on=USER_COL, how="inner")

print("Users:", df[USER_COL].nunique())
print("Total posts:", len(df))
print("Median posts/user:", df.groupby(USER_COL).size().median())
print(df.drop_duplicates(USER_COL)["depression"].value_counts())

Users: 165
Total posts: 6420
Median posts/user: 5.0
depression
0    105
1     60
Name: count, dtype: int64


In [7]:
df = tl.merge(users[[USER_COL, "depression", "anxiety"]], on=USER_COL, how="inner")

print("Users:", df[USER_COL].nunique(), "| Posts:", len(df))
print(df.drop_duplicates(USER_COL)["depression"].value_counts())

per_user = df.groupby(USER_COL).agg(n=(TIME_COL, "count"), label=("depression", "first"))

print("\nMin tweets -> users kept, depression label split")
for m in [3, 5, 10, 20, 30]:
    sub = per_user[per_user["n"] >= m]
    print(m, "->", len(sub), sub["label"].value_counts().to_dict())

Users: 165 | Posts: 6420
depression
0    105
1     60
Name: count, dtype: int64

Min tweets -> users kept, depression label split
3 -> 107 {0: 62, 1: 45}
5 -> 86 {0: 51, 1: 35}
10 -> 67 {0: 40, 1: 27}
20 -> 48 {0: 26, 1: 22}
30 -> 29 {1: 16, 0: 13}


In [ ]:
df["date"] = pd.to_datetime(df["date"])

print("Authors:", df["author"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

counts = df["author"].value_counts()
print("Authors with 2+ posts:", (counts >= 2).sum())
print("Authors with 5+ posts:", (counts >= 5).sum())
print(counts.head(10))


Authors: 1257
Date range: 2019-01-01 00:00:00 to 2019-04-20 00:00:00
Authors with 2+ posts: 0
Authors with 5+ posts: 0
author
LET_AFRICA_DIE       1
Lillie1990           1
P3nw00dx             1
throwaway146900      1
deeseypieces14       1
asdfderp1009         1
wizardslashmagi      1
fuckingburner1232    1
Girl-Here            1
ReallybigTomatos     1
Name: count, dtype: int64


In [ ]:
!wget -O ic_data.zip "https://zenodo.org/records/5716441/files/ic_data.zip?download=1"

--2026-10-19 08:42:13--  https://zenodo.org/records/5716441/files/ic_data.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.45.63, 188.185.42.113, 188.185.40.132, ...
Connecting to zenodo.org (zenodo.org)|188.185.45.63|:443... connected.
HTTP request sent, awaiting response... 404 NOT FOUND
2026-10-19 08:42:14 ERROR 404: NOT FOUND.

